## Bayes Optimal v Prompting for Combination Lock

In [1]:
import plotly.express as px
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np

In [42]:
style = "1"
results_path: str = f'../src/optimal_explorer/strategies/combination_lock/logs/game_results/style{style}.jsonl'

models = [
    "Gemini Pro 2.5",
    "DeepSeek R1",
    "Claude Opus 4",
    "Claude 3.5 Sonnet",
    "OpenAI o3",
]

model_ids = [
    "google/gemini-2.5-pro-preview",
    "deepseek/deepseek-r1-0528",
    "anthropic/claude-opus-4",
    "anthropic/claude-3.5-sonnet",
    "openai/o3",
]

In [43]:
results = []

with open(results_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        regret = [0 if z['feedback'] == [2, 2, 2] else 1 for z in data['history']] + [0] * (12 - len(data['history']))
        model = data['model']
        results.append({
            'game_id': data['game_id'],
            'model': model,
            'regret': regret,
            'length': len(data['history']),
            'style': style
        })
results_df = pd.DataFrame(results)

In [44]:
results_df.head(3)

,game_id,model,regret,length,style
0,21,anthropic/claude-3.5-sonnet,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",3,1
1,22,anthropic/claude-3.5-sonnet,"[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]",4,1
2,67,anthropic/claude-3.5-sonnet,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]",5,1


In [45]:
results_df = results_df.drop_duplicates(subset=['game_id', 'model'], keep='last')

In [46]:
len(results_df)

500

In [47]:
cumulative_regrets = []
error_bars = []

for mi, model in enumerate(models):
    model_df = results_df[results_df['model'] == model_ids[mi]]
    # Convert regret lists to numpy array for easier computation
    regret_array = np.array(model_df['regret'].values.tolist())
    
    # Calculate mean regret per turn
    model_regret = np.mean(regret_array, axis=0)
    cumulative_regret = np.cumsum(model_regret)
    cumulative_regrets.append(cumulative_regret)
    
    # Calculate standard error of the mean for each turn
    sem = np.std(regret_array, axis=0) / np.sqrt(len(model_df))
    cumulative_sem = np.cumsum(sem)
    error_bars.append(cumulative_sem)
    
    print(f'{model} cumulative regret: {cumulative_regret}')

Gemini Pro 2.5 cumulative regret: [1.   2.   2.99 3.87 4.62 5.24 5.74 6.1  6.31 6.43 6.53 6.63]
DeepSeek R1 cumulative regret: [1.   1.99 2.93 3.82 4.68 5.44 6.1  6.59 6.99 7.33 7.61 7.85]
Claude Opus 4 cumulative regret: [1.   1.99 2.98 3.92 4.79 5.53 6.12 6.51 6.81 7.07 7.27 7.45]
Claude 3.5 Sonnet cumulative regret: [1.   2.   2.99 3.94 4.85 5.64 6.24 6.69 7.05 7.29 7.48 7.62]
OpenAI o3 cumulative regret: [1.   2.   2.95 3.86 4.64 5.25 5.63 5.85 6.01 6.06 6.09 6.09]


In [48]:
# Use Plotly's Dark24 color set for darker colors
colors = [px.colors.qualitative.Dark24[i] for i in [1, 10, 6, 15, 19]]

# Set LaTeX font for all text elements
latex_font = dict(
    family="Latin Modern Roman, Times New Roman, serif",
    size=14,
    color="black"
)

fig = go.Figure()
for mi, model in enumerate(models):
    x_vals = list(range(1, 13))
    y_mean = cumulative_regrets[mi]
    y_err = error_bars[mi]
    color = colors[mi % len(colors)]

    # Add shaded error region (as a filled area)
    fig.add_trace(go.Scatter(
        x=x_vals + x_vals[::-1],
        y=(y_mean + y_err).tolist() + (y_mean - y_err)[::-1].tolist(),
        fill='toself',
        fillcolor=f'rgba{tuple(int(color.lstrip("#")[i:i+2], 16) for i in (0, 2, 4)) + (0.18,)}',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False,
        name=f"{model} ± SEM"
    ))

    # Add main mean curve, thicker
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=y_mean,
        mode='lines+markers',
        name=model,
        line=dict(width=4, color=color),
        marker=dict(size=6, color=color)
    ))

# Add y=x baseline as a dashed line
baseline_x = list(range(1, 13))
baseline_y = list(range(1, 13))
fig.add_trace(go.Scatter(
    x=baseline_x,
    y=baseline_y,
    mode='lines',
    name='Baseline',
    line=dict(color='black', width=2, dash='dash'),
    showlegend=True
))

fig.update_layout(
    width=600,
    height=470,
    title=dict(
        text='',
        font=latex_font
    ),
    xaxis_title="Episode (Symbol Combo-Lock)",
    yaxis_title="Cumulative Regret",
    font=latex_font,
    xaxis=dict(
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        mirror=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    yaxis=dict(
        range=[1, 12],
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        mirror=True,
        side='right',  # default, but we want ticks on both sides
        showticksuffix='all',
        showticklabels=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    yaxis2=dict(
        overlaying='y',
        side='right',
        tickmode='linear',
        dtick=1,
        ticks='outside',
        showline=True,
        showticklabels=True,
        title_font=latex_font,
        tickfont=latex_font
    ),
    legend=dict(
        title='',
        x=0.03,  # left edge, inside plot
        y=0.97,  # top edge, inside plot
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='black',
        borderwidth=1,
        font=latex_font
    ),
    template='plotly_white'
)

# Add yaxis2 to all traces so ticks show on both sides
for trace in fig.data:
    trace.update(yaxis='y')

In [49]:
fig.show()

In [50]:
# TODOs

# Bayes-optimal baseline
# Error bars
# Efficiency (tokens per game for each model per episode)